# Independent face partitions and degrees

Skeletal enrichment leaves the macro mesh fixed. Each face may have its own partition and polynomial degrees. Here primal local submeshes have five subdivisions, while trace faces have two subdivisions.

The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/b8f57cd39f6c0787da68c130b8d80b5fb46ce2cea14eb6da79be02791b5bf1eb/03_skeleton_hp-companion.zip"
COMPANION_SHA256 = "b8f57cd39f6c0787da68c130b8d80b5fb46ce2cea14eb6da79be02791b5bf1eb"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/03_skeleton_hp.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pymhm import TriangleMesh, SkeletonSpace, FaceSpace
from examples.formulations.application import brinkman as solve_brinkman
from examples.formulations.application import elasticity as solve_elasticity


## Declare the local and global equations

The application writes local A, B and C blocks with the shared FEM kernels,
then supplies them to the generic assembler. Primal coefficients satisfy

$$
\begin{aligned}
a_T(p,v)&=(K\nabla p,\nabla v)_T,\\
L_T(v)&=(f,v)_T+\sum_j\alpha_jv(x_j),\\
B_T\lambda&=\langle\lambda,v\rangle_{\partial T},\\
C_Tp&=-\langle p,\mu\rangle_{\partial T}.
\end{aligned}
$$

Signs follow the fixed macroface normals. RT0 instead uses flux, pressure and
private boundary-pressure coefficients, including the local normal-flux
constraint. The global `Equation` supplies the signed Dirichlet moments and
retains each constant pressure mode. Pure Neumann data add the physical pressure
integral through an explicit constraint. `define_darcy` returns equations;
`assemble`, the global solve and field interpretation are separate stages.


In [ ]:
from pymhm import assemble, SolverConfig
from examples.formulations.darcy import define_darcy, pressure_constraints, recover_darcy

mesh = TriangleMesh.unit_square(2)
faces = tuple((FaceSpace((0.0, 0.4, 1.0), (i % 2, (i + 1) % 2)) for i in range(len(mesh.faces))))
skeleton = SkeletonSpace(mesh, faces)
result_definition = define_darcy(
    mesh, skeleton=skeleton, local_refinement=5, dirichlet=lambda x: 1 + x[:, 0] - 2 * x[:, 1]
)
result_system = assemble(result_definition.problem)
result = recover_darcy(
    result_definition,
    result_system,
    result_system.solve(constraints=pressure_constraints(result_definition, result_system)),
)
assert result.l2_error(lambda x: 1 + x[:, 0] - 2 * x[:, 1]) < 1e-11
print(
    "macro cells",
    len(mesh.cells),
    "trace DOFs",
    skeleton.size,
    "patch error",
    result.l2_error(lambda x: 1 + x[:, 0] - 2 * x[:, 1]),
)
for face, space in zip(mesh.faces, faces):
    a, b = mesh.points[face]
    cuts = a + np.array(space.breaks)[:, None] * (b - a)
    plt.plot(cuts[:, 0], cuts[:, 1], "k.-")
plt.title("Macro mesh edges and skeletal subdivision nodes")
plt.axis("equal")
plt.show()


The trace/local inf-sup condition still matters. Unlimited face enrichment with fixed low-dimensional local spaces can make the saddle matrix singular. The RT0 local backend requires constant trace segments aligned with its fine boundary faces.